In [ ]:
using ITensors, ITensorMPS, LinearAlgebra, HDF5
ITensors.disable_threaded_blocksparse() #Disables block sparse multithreading
BLAS.set_num_threads(3) #Enables multithreading with BLAS

### Initial Parameters ###
N = 15 #Number of lattice sites per dimension
d = 1 #Number of spatial dimensions
Dim = 10 #Truncated local Hilbert space dimension
a = 1.0 #Lattice spacing

n_0 = round(Int, ((N-1)/2)) #Index of the point at the center of the lattice (0-based indexing).

mass = 0.6
m0 = 1.0 #Basis frequency
l = 0.1 #phi^4 coupling strength

### Field Operator $\phi(\mathbf{x})$ and $\pi(\mathbf{x})$ ###
function a_matrix(D)
    A = zeros(D, D)
    for n in 1:D-1
        A[n, n+1] = sqrt(n)
    end
    return A
end

phi_matrix = D -> (a_matrix(D) + a_matrix(D)')/sqrt(2*m0)
pi_matrix = D -> im*sqrt(m0/2)*(a_matrix(D)' - a_matrix(D))

ITensors.op(::OpName"phi", ::SiteType"Boson", D::Int) = phi_matrix(D)

ITensors.op(::OpName"phi2", ::SiteType"Boson", D::Int) = phi_matrix(D)^2

ITensors.op(::OpName"phi4", ::SiteType"Boson", D::Int) = phi_matrix(D)^4

ITensors.op(::OpName"pi", ::SiteType"Boson", D::Int) = pi_matrix(D)

ITensors.op(::OpName"pi2", ::SiteType"Boson", D::Int) = pi_matrix(D)^2

### Hamiltonian ###
H0 = OpSum() #Non-interacting Hamiltonian OpSum
for x in 1:N
    global H0 += a^d/2, "pi2", x
    global H0 += a^d * d/a^2, "phi2", x
    if x < N
        global H0 -= a^d * 1/a^2, "phi", x, "phi", x+1
    else
        global H0 -= a^d * 1/a^2, "phi", x, "phi", 1
    end
    global H0 += a^d/2 * mass^2, "phi2", x
end

HInt = OpSum() #Interacting Hamiltonian OpSum
for x in 1:N
    global HInt += l/factorial(4) * a^d, "phi4", x
end

H_OS = H0 + HInt #Full Hamiltonian OpSum

sites = siteinds("Boson", N; dim=Dim); #Create ITensor sites

H = MPO(H_OS, sites); #Hamiltonian MPO

### Vacuum State MPS ###
psi0 = random_mps(sites;linkdims=10)
nsweeps = 30
maxdim = [50, 50, 100, 100, 200, 200]
cutoff = [1E-10]
energy, vac = dmrg(H,psi0;nsweeps,maxdim,cutoff)

EEC_MPS = h5open("Z:/Energy Correlator/1d_EEC_MPS/N=$N,a=$a,dim=$Dim,l=$l,m=$mass", "w")
write(EEC_MPS, "sites", sites)
write(EEC_MPS, "vac", vac)
close(EEC_MPS)

After sweep 1 energy=29.862792965183022  maxlinkdim=50 maxerr=5.38E-05 time=7.176
After sweep 2 energy=18.670643966693827  maxlinkdim=50 maxerr=1.39E-05 time=9.054
After sweep 3 energy=14.18819601713535  maxlinkdim=100 maxerr=1.87E-07 time=34.087
After sweep 4 energy=12.563758942985  maxlinkdim=100 maxerr=2.14E-07 time=41.462
After sweep 5 energy=11.783305415569739  maxlinkdim=200 maxerr=1.73E-09 time=135.332
After sweep 6 energy=11.29418255737363  maxlinkdim=200 maxerr=1.71E-09 time=160.813
After sweep 7 energy=11.054771004957642  maxlinkdim=200 maxerr=8.20E-10 time=155.660
After sweep 8 energy=10.959193755807746  maxlinkdim=200 maxerr=3.89E-10 time=145.328
After sweep 9 energy=10.921808450896183  maxlinkdim=200 maxerr=2.06E-10 time=130.589
After sweep 10 energy=10.908951447518639  maxlinkdim=187 maxerr=9.99E-11 time=103.670
After sweep 11 energy=10.903733008949333  maxlinkdim=169 maxerr=1.00E-10 time=84.739
After sweep 12 energy=10.901361676136247  maxlinkdim=144 maxerr=9.87E-11 time

(10.899224654415022, MPS
[1] ((dim=8|id=718|"Link,l=1"), (dim=10|id=938|"Boson,Site,n=1"))
[2] ((dim=23|id=899|"Link,l=2"), (dim=10|id=524|"Boson,Site,n=2"), (dim=8|id=718|"Link,l=1"))
[3] ((dim=10|id=748|"Boson,Site,n=3"), (dim=34|id=76|"Link,l=3"), (dim=23|id=899|"Link,l=2"))
[4] ((dim=10|id=377|"Boson,Site,n=4"), (dim=42|id=399|"Link,l=4"), (dim=34|id=76|"Link,l=3"))
[5] ((dim=10|id=384|"Boson,Site,n=5"), (dim=47|id=609|"Link,l=5"), (dim=42|id=399|"Link,l=4"))
[6] ((dim=10|id=165|"Boson,Site,n=6"), (dim=49|id=89|"Link,l=6"), (dim=47|id=609|"Link,l=5"))
[7] ((dim=10|id=897|"Boson,Site,n=7"), (dim=50|id=715|"Link,l=7"), (dim=49|id=89|"Link,l=6"))
[8] ((dim=10|id=429|"Boson,Site,n=8"), (dim=50|id=819|"Link,l=8"), (dim=50|id=715|"Link,l=7"))
[9] ((dim=10|id=593|"Boson,Site,n=9"), (dim=49|id=858|"Link,l=9"), (dim=50|id=819|"Link,l=8"))
[10] ((dim=10|id=219|"Boson,Site,n=10"), (dim=47|id=902|"Link,l=10"), (dim=49|id=858|"Link,l=9"))
[11] ((dim=10|id=772|"Boson,Site,n=11"), (dim=42|id=847|